In [25]:
import os
import re
import cv2
import timm
import torch
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm.auto import tqdm

import albumentations as A
from albumentations.pytorch import ToTensorV2

from torch.utils.data import Dataset, DataLoader

In [26]:
CFG = {
    "batch_size": 32,
    "num_classes": 3,
    "num_workers": 0,
    "device": "cuda"
}

DATA_DIR = Path("../data")
TEST_DIR = DATA_DIR / "test"

WEIGHT_DIR = Path("../weights")

In [27]:
def natural_sort_key(path):
    return [
        int(x) if x.isdigit() else x.lower()
        for x in re.split(r"(\d+)", path.name)
    ]


test_files = list(TEST_DIR.glob("*"))
test_files.sort(key=natural_sort_key)


class TestDataset(Dataset):

    def __init__(self, files, transforms=None):

        self.files = files
        self.transforms = transforms

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):

        img_path = self.files[idx]

        image = cv2.imread(str(img_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transforms:
            image = self.transforms(image=image)["image"]

        return image

In [28]:
test_dataset = TestDataset(
    test_files
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print(len(test_dataset))

1458


In [29]:
def predict_5fold(model_name, weight_pattern, img_size):

    transform = A.Compose([
        A.LongestMaxSize(max_size=img_size),
        A.PadIfNeeded(
            min_height=img_size,
            min_width=img_size,
            border_mode=cv2.BORDER_CONSTANT
        ),
        A.Normalize(),
        ToTensorV2()
    ])

    test_dataset = TestDataset(
        test_files,
        transform
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=CFG["batch_size"],
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )

    probs = np.zeros((len(test_dataset), 3), dtype=np.float32)

    softmax = torch.nn.Softmax(dim=1)

    for fold in range(5):

        print("=" * 50)
        print(f"{model_name} | Fold {fold}")

        model = timm.create_model(
            model_name,
            pretrained=False,
            num_classes=3
        ).to(CFG["device"])

        weight_path = WEIGHT_DIR / weight_pattern.format(fold=fold)

        print(weight_path)

        model.load_state_dict(
            torch.load(
                weight_path,
                map_location=CFG["device"]
            )
        )

        model.eval()

        fold_probs = []

        with torch.no_grad():

            for images in tqdm(test_loader):

                images = images.to(CFG["device"])

                outputs = model(images)

                outputs = softmax(outputs)

                fold_probs.append(outputs.cpu().numpy())

        fold_probs = np.concatenate(fold_probs)

        probs += fold_probs

    probs /= 5

    return probs

In [30]:
convnext_probs = predict_5fold(
    model_name="convnextv2_tiny.fcmae_ft_in22k_in1k",
    weight_pattern="best_fold{fold}.pth",
    img_size=384
)

convnextv2_tiny.fcmae_ft_in22k_in1k | Fold 0
..\weights\best_fold0.pth


100%|██████████| 46/46 [00:25<00:00,  1.79it/s]


convnextv2_tiny.fcmae_ft_in22k_in1k | Fold 1
..\weights\best_fold1.pth


100%|██████████| 46/46 [00:16<00:00,  2.83it/s]


convnextv2_tiny.fcmae_ft_in22k_in1k | Fold 2
..\weights\best_fold2.pth


100%|██████████| 46/46 [00:16<00:00,  2.82it/s]


convnextv2_tiny.fcmae_ft_in22k_in1k | Fold 3
..\weights\best_fold3.pth


100%|██████████| 46/46 [00:16<00:00,  2.82it/s]


convnextv2_tiny.fcmae_ft_in22k_in1k | Fold 4
..\weights\best_fold4.pth


100%|██████████| 46/46 [00:16<00:00,  2.81it/s]


In [31]:
swin_probs = predict_5fold(
    model_name="swinv2_tiny_window16_256",
    weight_pattern="swin_best_fold{fold}.pth",
    img_size=256
)

swinv2_tiny_window16_256 | Fold 0
..\weights\swin_best_fold0.pth


100%|██████████| 46/46 [00:11<00:00,  4.08it/s]


swinv2_tiny_window16_256 | Fold 1
..\weights\swin_best_fold1.pth


100%|██████████| 46/46 [00:11<00:00,  4.09it/s]


swinv2_tiny_window16_256 | Fold 2
..\weights\swin_best_fold2.pth


100%|██████████| 46/46 [00:11<00:00,  4.07it/s]


swinv2_tiny_window16_256 | Fold 3
..\weights\swin_best_fold3.pth


100%|██████████| 46/46 [00:11<00:00,  4.05it/s]


swinv2_tiny_window16_256 | Fold 4
..\weights\swin_best_fold4.pth


100%|██████████| 46/46 [00:11<00:00,  4.07it/s]


In [32]:
ALL_MODEL_PROBS = {
    "ConvNeXt": convnext_probs,
    "Swin": swin_probs
}

In [35]:
ENSEMBLES = {

    "A_50_50_S": {
        "ConvNeXt":0.50,
        "Swin":0.50
    },

    "B_40_60_S": {
        "ConvNeXt":0.40,
        "Swin":0.60
    },

    "C_30_70_S": {
        "ConvNeXt":0.30,
        "Swin":0.70
    },

    "D_20_80_S": {
        "ConvNeXt":0.20,
        "Swin":0.80
    },

    "E_60_40_S": {
        "ConvNeXt":0.60,
        "Swin":0.40
    }

}


In [36]:
sample_submission = pd.read_csv("../data/submission.csv")

for name, weights in ENSEMBLES.items():

    print("=" * 60)
    print(name)

    final_probs = np.zeros_like(convnext_probs)

    for model_name, weight in weights.items():

        final_probs += (
            weight *
            ALL_MODEL_PROBS[model_name]
        )

    predictions = final_probs.argmax(axis=1)

    submission = sample_submission.copy()

    submission["predicted"] = predictions

    save_path = (
        f"../output/submission_{name}.csv"
    )

    submission.to_csv(
        save_path,
        index=False
    )

    print(save_path)

    print(submission["predicted"].value_counts().sort_index())

    print()


A_50_50_S
../output/submission_A_50_50_S.csv
predicted
0    517
1    202
2    739
Name: count, dtype: int64

B_40_60_S
../output/submission_B_40_60_S.csv
predicted
0    516
1    205
2    737
Name: count, dtype: int64

C_30_70_S
../output/submission_C_30_70_S.csv
predicted
0    516
1    205
2    737
Name: count, dtype: int64

D_20_80_S
../output/submission_D_20_80_S.csv
predicted
0    521
1    204
2    733
Name: count, dtype: int64

E_60_40_S
../output/submission_E_60_40_S.csv
predicted
0    514
1    203
2    741
Name: count, dtype: int64

